# Limpieza de datos y estadísticas descriptivas — AlertaSegura Perú

Sprint 2 · Data Analysis

1. Corre el pipeline de limpieza (`scripts/limpieza_datos.py`) sobre
   `data/reportes_sucios.csv` — una versión con nulos, duplicados y
   formatos inconsistentes inyectados a propósito para probar el script.
2. Calcula las primeras estadísticas descriptivas sobre el dataset ya
   limpio.

## 1. Limpieza de datos

In [ ]:
import sys
sys.path.append("../scripts")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from limpieza_datos import limpiar

sns.set_theme(style="whitegrid")

sucio = pd.read_csv("../data/reportes_sucios.csv")
print("Filas:", len(sucio))
sucio.isnull().sum()

In [ ]:
print("Duplicados exactos (excluyendo id):", sucio.duplicated(subset=[c for c in sucio.columns if c != "id"]).sum())
sucio["distrito"].unique()[:10]

In [ ]:
reportes, reporte_limpieza = limpiar(sucio)

print(f"Filas antes: {reporte_limpieza['filas_iniciales']}  →  Filas después: {reporte_limpieza['filas_finales']}")
reporte_limpieza

In [ ]:
# Verificación: ya no quedan nulos ni duplicados, y el texto está normalizado
assert reportes.isnull().sum().sum() == 0
assert reportes.duplicated(subset=[c for c in reportes.columns if c != "id"]).sum() == 0
sorted(reportes["distrito"].unique())

In [ ]:
reportes["created_at"] = pd.to_datetime(reportes["created_at"])
reportes.to_csv("../data/reportes_limpios.csv", index=False)
reportes.head()

## 2. Estadísticas descriptivas

In [ ]:
reportes.describe(include="all")

In [ ]:
print("Total de reportes:", len(reportes))
print("Distritos distintos:", reportes["distrito"].nunique())
print("Categorías distintas:", reportes["categoria"].nunique())
print("Rango de fechas:", reportes["created_at"].min(), "→", reportes["created_at"].max())

In [ ]:
# Reportes promedio por distrito, y distrito con más/menos reportes
conteo_distrito = reportes["distrito"].value_counts()
print("Promedio de reportes por distrito:", round(conteo_distrito.mean(), 1))
print("Distrito con más reportes:", conteo_distrito.idxmax(), f"({conteo_distrito.max()})")
print("Distrito con menos reportes:", conteo_distrito.idxmin(), f"({conteo_distrito.min()})")

In [ ]:
# Distribución de categorías (%)
(reportes["categoria"].value_counts(normalize=True) * 100).round(1)

In [ ]:
# Tasa de verificación global
(reportes["estado"].value_counts(normalize=True) * 100).round(1)

In [ ]:
# Tabla cruzada: categoría x estado (conteo)
pd.crosstab(reportes["categoria"], reportes["estado"])

In [ ]:
reportes["hora"] = reportes["created_at"].dt.hour
reportes.groupby("categoria")["hora"].agg(["mean", "median", "std"]).round(1)

## Próximos pasos (Sprint 3)

- Generar un dataset sintético más grande y realista (150-300 registros).
- Primeros gráficos formales: reportes por distrito y por tipo de incidente
  (ya cubiertos parcialmente en `01_exploracion_inicial.ipynb`, se refinan aquí).